In [88]:
import pandas as pd
import matplotlib.pyplot as plt
import scipy as stats
import seaborn as sns
import numpy as np

# 출력 짤림 방지
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [89]:
# 데이터 불러오기
df = pd.read_csv('./data/Courses.csv', parse_dates=['start_time_DI', 'last_event_DI'])
df.head()


,index,course_id,userid_DI,registered,viewed,explored,certified,final_cc_cname_DI,LoE_DI,YoB,gender,grade,start_time_DI,last_event_DI,nevents,ndays_act,nplay_video,nchapters,nforum_posts,roles,incomplete_flag
0,0,HarvardX/CB22x/2013_Spring,MHxPC130442623,1,0,0,0,United States,NaN,NaN,NaN,0,2012-12-19,2013-11-17,NaN,9.0,NaN,NaN,0,NaN,1.0
1,1,HarvardX/CS50x/2012,MHxPC130442623,1,1,0,0,United States,NaN,NaN,NaN,0,2012-10-15,NaT,NaN,9.0,NaN,1.0,0,NaN,1.0
2,2,HarvardX/CB22x/2013_Spring,MHxPC130275857,1,0,0,0,United States,NaN,NaN,NaN,0,2013-02-08,2013-11-17,NaN,16.0,NaN,NaN,0,NaN,1.0
3,3,HarvardX/CS50x/2012,MHxPC130275857,1,0,0,0,United States,NaN,NaN,NaN,0,2012-09-17,NaT,NaN,16.0,NaN,NaN,0,NaN,1.0
4,4,HarvardX/ER22x/2013_Spring,MHxPC130275857,1,0,0,0,United States,NaN,NaN,NaN,0,2012-12-19,NaT,NaN,16.0,NaN,NaN,0,NaN,1.0


In [90]:
# 원본데이터 결측치 개수, 비율
display(pd.DataFrame({
    'sum': df.isna().sum(),
    'ratio': df.isna().mean() * 100
}).sort_values('ratio', ascending=False).reset_index())

,index,sum,ratio
0,roles,641138,100.000000
1,incomplete_flag,540977,84.377622
2,nplay_video,457530,71.362172
3,nchapters,258753,40.358394
4,nevents,199151,31.062111
5,last_event_DI,178954,27.911932
6,ndays_act,162743,25.383459
7,LoE_DI,106008,16.534350
8,YoB,96605,15.067739
9,gender,86806,13.539363


In [91]:
# 전처리용 데이터 셋 생성
pre = df.copy()

In [92]:
# 중복행 확인
len(pre.duplicated())

641138

In [93]:
# 의미없는 컬럼 제거
pre = pre.drop(columns=['index', 'roles'])


In [94]:
### 결측치 대체 & 행 제거

In [95]:
# course_id 이상치 
    # 확인 후 이상치의 공통점 찾아서 처리여부 파악 (직접 찍어보고 확인)
    
pre[['course_code', 'course', 'course_year']] = pre['course_id'].str.split('/', expand=True) # expand=True df로 반환(열로세우기)

# CS50x는 분류 상 '상시 강의' / '기간 강의' 중 본 프로젝트의 목표에 맞지 않는 분류라 분석에 사용하지 않기로 결정
pre = pre[~pre['course_id'].str.contains('CS50x')].copy()   #~로 아닌 애들만 ㄱㄱ

print(pre['course'].value_counts())

print(pre['course_year'].value_counts())

course
6.00x     124446
6.002x     63046
ER22x      57406
PH207x     41592
PH278x     39602
8.02x      31048
CB22x      30002
14.73x     27870
7.00x      21009
3.091x     20354
8.MReV      9477
2.01x       5665
Name: count, dtype: int64
course_year
2013_Spring    298691
2012_Fall      163349
2013_Summer      9477
Name: count, dtype: int64


In [96]:
# 나라 컬럼명 변경
pre = pre.rename(columns={'final_cc_cname_DI':'country'})

In [97]:
# n~ 컬럼
n_cols = ['nplay_video', 'nevents', 'ndays_act', 'nchapters']
pre[n_cols] = pre[n_cols].fillna(0)
# 근거 : nplay_video, 0이 존재하지 않고, '시청시간이 없는 값'이 모두 NaN 처리가 되었다는 논리적 근거 O

In [98]:
# 110531 <<< 제거대상 
pre = pre.drop(pre[(pre['viewed'] == 0) & ((pre['nchapters'] > 0) | (pre['nevents'] > 0))].index)


In [99]:
# 논리적 이상치
    # nchapters값이 1인데 nan값으로 입력되었을 리 X ∴ 이상치
    # nplay_video의 기준이 클릭/완강인지 명시 X, 
    # But viewed 컬럼이 따로 존재하는 것으로 nplay_video가 완강 기준일 것이라는 근거로 분석을 진행했다.
    # nplay_video가 완강 기준일 것이라는 근거로 분석을 진행했다.
nonono = (pre['viewed'] == 0) & ((pre['nchapters'] > 0) | (pre['nevents'] > 0))

In [100]:
# start_time_DI (결측 71.36) -> 대신 ndays_act 이용하기?
# 음수가 나오는 애들 제거 후 한계점 감안하고 사용하라는 튜터님 피드백 있었음


In [101]:
# 아니면 ndays_act를 이용하는 방법 (몇일 이상 접속한 사람으로 구간 나눠서?) -> 그 기준은?
# 조사 필요할 듯
# ndays_act를 가지고 몇일 이상 접속해야 찍먹러가 아닌 기준에 대해서 명확하게 하고 싶은데

In [102]:
# 찍먹과 학습의지 수강생을 나눌 수 있을까?

ac_df = df[df['viewed'] == 1].copy()
viewed_rate = (df['viewed'] == 1).mean() * 100
viewed_rate # 62.42

np.float64(62.429929281995456)

In [103]:
# 파생컬럼 생성

# 학생들의 나이(age)
pre['age'] = pre['start_time_DI'].dt.year - pre['YoB']

# YoB 결측치를 0으로 채우기
pre['YoB'] = pre['YoB'].fillna(0)

# 최소 가입 나이 (13세)
pre.loc[pre['age'] < 13, 'age']

# 퍼널 단계 컬럼(step): 각 학생 별 진행 단계
pre['step'] = np.select(
    [
        pre['certified'] ==1,
        pre['explored'] == 1,
        pre['viewed'] == 1,
        pre['registered'] == 1,
    ],
    [
        'Certified',
        'Explored',
        'Viewed',
        'Registered'
    ],
    default='None'
)

In [104]:
# Unknown
un_cols = ['gender', 'LoE_DI', 'YoB']
pre[un_cols] = pre[un_cols].fillna('Unknown')

In [106]:
pre['age'].describe()

count    299277.000000
mean         27.279991
std           8.893779
min           0.000000
25%          21.000000
50%          25.000000
75%          30.000000
max          82.000000
Name: age, dtype: float64

In [ ]:
# 퍼널 분석을 위한 탐색
    # 결국 우리 분석 대상은
    # 이탈자 / 반복학습생이 아닐까?
    # '끝까지 수료할 가능성'이 있는 사람들은 끌어모으고 << 잠재고객? 가치 파악? 수익성 ↑?
    # '이탈할 가능성' = 리스크는 줄이는 게 맞으니까 << 사실 이게 더 중요하긴 함 : 놓친 사람
    
# 이탈 이유가 결국 수강 의지가 있는 부류 / 없는데 이탈하는 부류 < 이 문제점을 파악해야
# 등록 - 첫수강에서 막히는지(진입 or 환경) / 수강 - 유지에서 막히는지(난이도)
# 찍먹러인지 중도포기인지 반복학습자? 인지
# 근데 이탈하면 어디서 많이 이탈하는지가 중요 -> 그래서 우리가 퍼널 별 구간 이탈자 탐색 < 이걸 하는거고 음음 굿

# 찍먹러는 ndays_act가 적은 사람으로 판단 가능? < 조사 필요
# 중도포기는 certified로 잡고 싶었는데 상당수가 수료 못해서 분류 기준이 비상이고
# 

685